In [1]:
import pandas as pd
import numpy as np

### Cutoffs

In [2]:
english = pd.read_csv('../../data/process/english/adverse.csv')
english.groupby('source')['score'].mean()

source
2019-train           0.044396
2020-test-google     0.553397
2020-test-yandex     0.533863
2020-train           0.171109
2020-valid-google    0.579775
2020-valid-yandex    0.565616
prev-test            0.085698
Name: score, dtype: float64

In [3]:
foreign = pd.read_csv('../../data/process/foreign/adverse.csv')
foreign.groupby('source')['score'].mean()

source
2019-train    0.010442
2020-test     0.213982
2020-train    0.028840
2020-valid    0.233643
prev-test     0.017434
Name: score, dtype: float64

In [4]:
foreign = pd.read_csv('../../data/process/foreign/adverse_valid.csv')
foreign.groupby('source')['score'].mean()

source
2020-test     0.020560
2020-valid    0.028979
Name: score, dtype: float64

In [5]:
subtitle = pd.read_csv('../../data/process/subtitle/adverse.csv')
subtitle.groupby('source')['score'].mean()

source
2020-test     0.807315
2020-valid    0.816981
subtitle      0.020679
Name: score, dtype: float64

### English

In [6]:
train = pd.read_csv('../../data/process/english/train_english.csv')
valid = pd.read_csv('../../data/process/english/valid_english.csv')
test = pd.read_csv('../../data/process/english/test_english.csv')
score = pd.read_csv('../../data/process/english/adverse.csv')
score = score.rename(columns={'score':'weight'})

/usr/local/lib/python3.6/dist-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (0) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [7]:
train['id'] = train['id'].astype(str)
score['id'] = score['id'].astype(str)

In [8]:
train.shape, valid.shape, test.shape, score.shape

((802105, 5), (16000, 6), (127624, 5), (945729, 4))

In [9]:
train = train.merge(score, on=['id','source','lang'])

In [10]:
min_val, max_val = train['weight'].min(), train['weight'].max()
train['weight'] = (train['weight'] - min_val) / (max_val - min_val) 
valid['weight'] = 1.
test['weight'] = 1.

In [11]:
train.shape, valid.shape, test.shape, score.shape

((802105, 6), (16000, 7), (127624, 6), (945729, 4))

In [12]:
train['weight'].describe().round(3)

count    802105.000
mean          0.081
std           0.128
min           0.000
25%           0.011
50%           0.030
75%           0.085
max           1.000
Name: weight, dtype: float64

In [13]:
train.to_csv('../../data/process/english/train_english.csv', index=False)
valid.to_csv('../../data/process/english/valid_english.csv', index=False)
test.to_csv('../../data/process/english/test_english.csv', index=False)

### Foreign

In [14]:
train = pd.read_csv('../../data/process/foreign/train_foreign.csv')
valid = pd.read_csv('../../data/process/foreign/valid_foreign.csv')
test = pd.read_csv('../../data/process/foreign/test_foreign.csv')
score1 = pd.read_csv('../../data/process/foreign/adverse.csv')
score2 = pd.read_csv('../../data/process/foreign/adverse_valid.csv')
score1 = score1.rename(columns={'score':'weight'})
score2 = score2.rename(columns={'score':'weight'})
score = score1.append(score2)

In [15]:
train['id'] = train['id'].astype(str)
valid['id'] = valid['id'].astype(str)
test['id'] = test['id'].astype(str)
score['id'] = score['id'].astype(str)
score2['id'] = score2['id'].astype(str)

In [16]:
train.shape, valid.shape, test.shape, score.shape

((2898334, 5), (47950, 6), (382367, 5), (3336651, 4))

In [17]:
train = train.merge(score, on=['id','source','lang'])
valid = valid.merge(score2, on=['id','source','lang'])
test = test.merge(score, on=['id','source','lang'])

In [18]:
train.shape, valid.shape, test.shape, score.shape

((2901332, 6), (47950, 7), (382367, 6), (3336651, 4))

In [19]:
train.to_csv('../../data/process/foreign/train_foreign.csv', index=False)
valid.to_csv('../../data/process/foreign/valid_foreign.csv', index=False)
test.to_csv('../../data/process/foreign/test_foreign.csv', index=False)

### Subtitle

In [32]:
subtitle = pd.read_csv('../../data/process/subtitle/subtitle.csv')
score = pd.read_csv('../../data/process/subtitle/adverse.csv')
score = score.rename(columns={'score':'weight'})

In [33]:
subtitle.shape

(636310, 7)

In [34]:
subtitle = subtitle.merge(score, on=['id','source','lang'])

In [35]:
subtitle.shape

(636310, 8)

In [36]:
subtitle.to_csv('../../data/process/subtitle/subtitle.csv', index=False)

### Append

### Subset

In [32]:
english = pd.read_csv('../../data/process/english/train_english.csv')
foreign = pd.read_csv('../../data/process/foreign/train_foreign.csv')
subtitle = pd.read_csv('../../data/process/subtitle/subtitle.csv')

/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (0) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [33]:
data = english.append(foreign).append(subtitle)
data = data.sample(frac=1., random_state=2017)
data = data.reset_index(drop=True)

In [34]:
data['weight'] = 1.
data.loc[data['source'] == '2020-train','weight'] = 4.

In [35]:
data.shape, data['toxic'].mean()

((4339747, 8), 0.24602978007704135)

In [36]:
data.groupby('source')['toxic'].agg(['mean','count'])

,mean,count
source,,
2019-train,0.274775,2069417
2020-train,0.095416,1568301
prev-test,1.000000,65719
subtitle,0.445886,636310


In [37]:
data.groupby('lang')['toxic'].agg(['mean','count'])

,mean,count
lang,,
en,0.228015,802105
es,0.262368,639861
fr,0.249473,594955
it,0.251257,585931
pt,0.246059,570001
ru,0.237682,551254
tr,0.251855,595640


In [38]:
data.weight.value_counts()

1.0    2771446
4.0    1568301
Name: weight, dtype: int64

In [39]:
data = data.drop(['weight_x','weight_y'], axis=1)

In [40]:
data.head()

,id,source,lang,comment_text,toxic,weight
0,259638,subtitle,es,"Y todas las noches, se sentaba frente a la ven...",1,1.0
1,249513,subtitle,ru,"Мальчики носят мои коктейли на склад и они, ка...",0,1.0
2,d52a9dd76a7a9bf2,2020-train,it,AfD nomination of Beasly and Tha Foolfatha\nUn...,0,4.0
3,564474,subtitle,pt,"Há um cartão na minha mesa para o Isaac Lahey,...",0,1.0
4,df73f82d6b41edd6,2020-train,en,)\n\nDoors are great but they are even better ...,0,4.0


In [41]:
data.to_csv('../../data/process/pseudo/train_combine.csv', index=False)

In [42]:
english = pd.read_csv('../../data/process/english/valid_english.csv')
foreign = pd.read_csv('../../data/process/foreign/valid_foreign.csv')

In [43]:
data = english.append(foreign)
data = data.sample(frac=1., random_state=2017)
data = data.reset_index(drop=True)

In [44]:
data['weight'] = 3 * data['original'] + 1

In [45]:
data['weight'].value_counts()

1    55950
4     8000
Name: weight, dtype: int64

In [46]:
data.to_csv('../../data/process/pseudo/valid_combine.csv', index=False)

In [47]:
english = pd.read_csv('../../data/process/english/test_english.csv')
foreign = pd.read_csv('../../data/process/foreign/test_foreign.csv')

In [48]:
data = english.append(foreign)
data = data.sample(frac=1., random_state=2017)
data = data.reset_index(drop=True)

In [49]:
data['weight'] = 3 * data['original'] + 1

In [50]:
data['weight'].value_counts()

1    446163
4     63828
Name: weight, dtype: int64

In [51]:
data.to_csv('../../data/process/pseudo/test_combine.csv', index=False)